In [1]:
# === Imports and configuration ===
import os
import glob
import time
import pandas as pd
import numpy as np
import requests

# Global variables
START_DATE = '2024-01-01'
END_DATE = '2024-12-31'

# Csv files paths
INPUT_DIR = "2024_capital_cities_data"
INPUT_PATTERN = os.path.join(INPUT_DIR, "2024_data_*.csv")

# output
OUTPUT_CSV = "solar_weather_all_cities.csv"

# API
API_SLEEP_S = 5            # segundos de pausa entre llamadas para no saturar
USE_METEO_CACHE = True     # cachear meteo por ciudad/año
CACHE_DIR = "meteo_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

In [3]:
# === City coordinates ===
d = {
    'city': [
        'Madrid', 'Paris', 'Berlin', 'Rome', 'Vienna', 'Brussels', 'Sofia',
        'Zagreb', 'Nicosia', 'Prague', 'Copenhagen', 'Tallinn', 'Helsinki',
        'Athens', 'Budapest', 'Dublin', 'Riga', 'Vilnius', 'Luxembourg',
        'Valletta', 'Amsterdam', 'Warsaw', 'Lisbon', 'Bucharest', 'Bratislava',
        'Ljubljana', 'Stockholm', 'London'
    ],
    'latitude': [
        40.4165, 48.85341, 52.52281, 41.89306, 48.2082, 50.8503, 42.6977,
        45.815, 35.1856, 50.0755, 55.6761, 59.437, 60.1699, 37.9838, 47.4979,
        53.3498, 56.9496, 54.6872, 49.6116, 35.8989, 52.3676, 52.2297,
        38.7223, 44.4268, 48.1486, 46.0569, 59.3293, 51.5074
    ],
    'longitude': [
        -3.70256, 2.3488, 13.408176, 12.48278, 16.3738, 4.3517, 23.3219,
        15.9819, 33.3823, 14.4378, 12.5683, 24.7536, 24.9384, 23.7275,
        19.0402, -6.2603, 24.1052, 25.2797, 6.1319, 14.5146, 4.9041,
        21.0122, -9.1393, 26.1025, 17.1077, 14.5058, 18.0686, -0.1278
    ]
}
coord_df = pd.DataFrame(d).set_index("city")
coord_df.head()

,latitude,longitude
city,,
Madrid,40.41650,-3.702560
Paris,48.85341,2.348800
Berlin,52.52281,13.408176
Rome,41.89306,12.482780
Vienna,48.20820,16.373800


In [4]:
# === Multipurpose functions ===
def city_from_filename(path):
    """Extract city exactly as appeas in the file"""
    name = os.path.basename(path)
    return name.replace("2024_data_", "").replace(".csv", "")

def output_has_city(path, city):
    """Checks if city is already in master csv to continue without duplicates"""
    if not os.path.exists(path):
        return False
    for chunk in pd.read_csv(path, chunksize=200_000):
        if "city" in chunk.columns and (chunk["city"] == city).any():
            return True
    return False

def append_csv(df, path):
    header = not os.path.exists(path)
    df.to_csv(path, mode="a", header=header, index=False)

def read_city_solar_csv(file):
    """Read irradiance csv file of a city and creates 1-min timestamp index."""
    df = pd.read_csv(file, sep=';')
    num_rows = len(df)
    times = pd.date_range(START_DATE, freq='1min', periods=num_rows).tz_localize('UTC')
    df = df.set_index(times)
    df.index.names = ['time']
    return df[['GHI', 'BHI', 'DHI']].reset_index()

In [5]:
# === Meteo API ===
def get_city_coords(city):
    if city not in coord_df.index:
        raise ValueError(f"Coordenadas no encontradas para ciudad: {city}")
    row = coord_df.loc[city]
    return float(row["latitude"]), float(row["longitude"])

def minute_index_for_year(start_date, periods):
    return pd.date_range(start_date, freq='1min', periods=periods, tz='UTC')

def fetch_meteo_city(city, start_date, end_date):
    """Descarga meteo para UNA ciudad y la expande a minuto; usa caché parquet."""
    lat, lon = get_city_coords(city)
    year = pd.Timestamp(start_date).year
    cache_path = os.path.join(CACHE_DIR, f"meteo_{city}_{year}.parquet")

    if USE_METEO_CACHE and os.path.exists(cache_path):
        dfh = pd.read_parquet(cache_path)
        # aseguramos que esté en UTC
        if dfh.index.tz is None:
            dfh.index = dfh.index.tz_localize("UTC")
        else:
            dfh.index = dfh.index.tz_convert("UTC")
        return dfh

    variable = "temperature_2m,wind_speed_10m"
    url = (f"https://archive-api.open-meteo.com/v1/era5"
           f"?latitude={lat}&longitude={lon}&hourly={variable}"
           f"&timezone=GMT&start_date={start_date}&end_date={end_date}")
    resp = requests.get(url).json()
    dfh = pd.DataFrame(resp["hourly"])  # hourly
    # Expandir cada registro horario en 60 minutos
    dfh = dfh.loc[dfh.index.repeat(60)].reset_index(drop=True)

    periods = 60 * 24 * (366 if pd.Timestamp(str(year)).is_leap_year else 365)
    idx = minute_index_for_year(start_date, periods)
    dfh["time"] = idx
    dfh["city"] = city
    dfh["lat"] = lat
    dfh["lon"] = lon

    dfh = dfh.set_index("time")
    if dfh.index.tz is None:
        dfh.index = dfh.index.tz_localize("UTC")
    else:
        dfh.index = dfh.index.tz_convert("UTC")

    if USE_METEO_CACHE:
        dfh.to_parquet(cache_path, index=True)
    time.sleep(API_SLEEP_S)
    return dfh

In [6]:
# === City processing ===
def process_one_city(file, output_csv=OUTPUT_CSV):
    city = city_from_filename(file)
    if output_has_city(output_csv, city):
        print(f"⏭️  Saltando {city}: ya en {output_csv}")
        return

    print(f"🔄 Procesando {city}")
    solar_df = read_city_solar_csv(file)
    meteo_df = fetch_meteo_city(city, START_DATE, END_DATE).reset_index()

    merged = solar_df.merge(meteo_df, on="time", how="inner").rename(columns={
        'GHI': 'ghi',
        'BHI': 'dni',     # tu CSV usa BHI -> DNI
        'DHI': 'dhi',
        'wind_speed_10m': 'wind_speed',
        'temperature_2m': 'temp_air'
    })
    # Conversión de unidades
    merged['ghi'] *= 60
    merged['dni'] *= 60
    merged['dhi'] *= 60
    merged['wind_speed'] *= (1000/3600)
    merged['city'] = city

    append_csv(merged, output_csv)
    print(f"✅ {city} añadido ({len(merged):,} filas)")

In [7]:
# === Batch running (memory limit) ===
def run_batch(target_cities=None):
    files = glob.glob(INPUT_PATTERN)
    if target_cities:
        target = set(target_cities)  # ya vienen capitalizadas en tus archivos
        files = [f for f in files if city_from_filename(f) in target]

    if not files:
        print("⚠️ No hay ficheros que procesar")
        return

    for f in sorted(files):
        try:
            process_one_city(f, OUTPUT_CSV)
        except Exception as e:
            print(f"❌ Error en {f}: {e}")

In [10]:
# run_batch()

# Example: batch
run_batch(["Madrid", "Paris", "Berlin", "Rome", "Vienna", "Brussels", "Sofia", 
 "Zagreb", "Nicosia", "Prague", "Copenhagen", "Tallinn", "Helsinki",
 "Athens", "Budapest", "Dublin", "Riga", "Vilnius", "Luxembourg",
 "Valletta", "Amsterdam", "Warsaw", "Lisbon", "Bucharest", "Bratislava",
 "Ljubljana", "Stockholm", "London"])

🔄 Procesando Amsterdam
✅ Amsterdam añadido (527,040 filas)
🔄 Procesando Athens
✅ Athens añadido (527,040 filas)
🔄 Procesando Berlin
✅ Berlin añadido (527,040 filas)
🔄 Procesando Bratislava
✅ Bratislava añadido (527,040 filas)
🔄 Procesando Brussels
✅ Brussels añadido (527,040 filas)
🔄 Procesando Bucharest
✅ Bucharest añadido (527,040 filas)
🔄 Procesando Budapest
✅ Budapest añadido (527,040 filas)
🔄 Procesando Copenhagen
✅ Copenhagen añadido (527,040 filas)
🔄 Procesando Dublin
✅ Dublin añadido (527,040 filas)
🔄 Procesando Helsinki
✅ Helsinki añadido (527,040 filas)
🔄 Procesando Lisbon
✅ Lisbon añadido (527,040 filas)
🔄 Procesando Ljubljana
✅ Ljubljana añadido (527,040 filas)
🔄 Procesando London
✅ London añadido (527,040 filas)
🔄 Procesando Luxembourg
✅ Luxembourg añadido (527,040 filas)
⏭️  Saltando Madrid: ya en solar_weather_all_cities.csv
🔄 Procesando Nicosia
✅ Nicosia añadido (527,040 filas)
🔄 Procesando Paris
✅ Paris añadido (527,040 filas)
🔄 Procesando Prague
✅ Prague añadido (527,

In [11]:
df_out = pd.read_csv(OUTPUT_CSV, nrows=10)
print(df_out.head())

                        time  ghi  dni  dhi  temp_air  wind_speed    city  \
0  2024-01-01 00:00:00+00:00  0.0  0.0  0.0       4.2    1.305556  Madrid   
1  2024-01-01 00:01:00+00:00  0.0  0.0  0.0       4.2    1.305556  Madrid   
2  2024-01-01 00:02:00+00:00  0.0  0.0  0.0       4.2    1.305556  Madrid   
3  2024-01-01 00:03:00+00:00  0.0  0.0  0.0       4.2    1.305556  Madrid   
4  2024-01-01 00:04:00+00:00  0.0  0.0  0.0       4.2    1.305556  Madrid   

       lat      lon  
0  40.4165 -3.70256  
1  40.4165 -3.70256  
2  40.4165 -3.70256  
3  40.4165 -3.70256  
4  40.4165 -3.70256  


In [2]:
"""
PVGIS calibration script for the 28 European capitals in the paper.

⚠️ IMPORTANT: This script generates the f_cal calibration factors based on the
UNCALIBRATED pipeline output (model yields from notebook 2 BEFORE applying f_cal).
Do NOT re-run this script after the calibration has been applied to the pipeline,
or you will overwrite the correct factors with k_cal ≈ 1.0.

Queries the PVGIS-SARAH3 API for each city using the same fixed-tilt configuration
as the manuscript (tilt = 0.76 * latitude + 3.1, south-facing, 14% system losses,
1 kWp reference, crystalline silicon), and computes the per-city calibration
coefficient f_cal,i = PVGIS_yield / model_yield.

Requirements: pandas, requests
Install: pip install pandas requests
"""

import pandas as pd
import requests
import time
from pathlib import Path

# -------------------------------------------------------------
# 1. CITY COORDINATES — from manuscript Appendix C
# -------------------------------------------------------------
CITIES = [
    # (country,         city,         latitude,  longitude)
    ("Austria",         "Vienna",     48.2082,  16.3738),
    ("Belgium",         "Brussels",   50.8503,   4.3517),
    ("Bulgaria",        "Sofia",      42.6977,  23.3219),
    ("Croatia",         "Zagreb",     45.8150,  15.9819),
    ("Cyprus",          "Nicosia",    35.1856,  33.3823),
    ("Czech Republic",  "Prague",     50.0755,  14.4378),
    ("Denmark",         "Copenhagen", 55.6761,  12.5683),
    ("Estonia",         "Tallinn",    59.4370,  24.7536),
    ("Finland",         "Helsinki",   60.1699,  24.9384),
    ("France",          "Paris",      48.8566,   2.3522),
    ("Germany",         "Berlin",     52.5200,  13.4050),
    ("Greece",          "Athens",     37.9838,  23.7275),
    ("Hungary",         "Budapest",   47.4979,  19.0402),
    ("Ireland",         "Dublin",     53.3498,  -6.2603),
    ("Italy",           "Rome",       41.9028,  12.4964),
    ("Latvia",          "Riga",       56.9496,  24.1052),
    ("Lithuania",       "Vilnius",    54.6872,  25.2797),
    ("Luxembourg",      "Luxembourg", 49.6116,   6.1319),
    ("Malta",           "Valletta",   35.8989,  14.5146),
    ("Netherlands",     "Amsterdam",  52.3676,   4.9041),
    ("Poland",          "Warsaw",     52.2297,  21.0122),
    ("Portugal",        "Lisbon",     38.7223,  -9.1393),
    ("Romania",         "Bucharest",  44.4268,  26.1025),
    ("Slovakia",        "Bratislava", 48.1486,  17.1077),
    ("Slovenia",        "Ljubljana",  46.0569,  14.5058),
    ("Spain",           "Madrid",     40.4168,  -3.7038),
    ("Sweden",          "Stockholm",  59.3293,  18.0686),
    ("United Kingdom",  "London",     51.5074,  -0.1278),
]

# -------------------------------------------------------------
# 2. MODEL YIELDS — from UNCALIBRATED Appendix C, Table C1
#    Computed as: Yearly AC Energy output (MWh) / 50 MW * 1000
#    (specific yield kWh/kWp/year for ILR=1.0, BEFORE f_cal applied)
# -------------------------------------------------------------
MODEL_YIELDS_MWh = {
    "Vienna":      52641,
    "Brussels":    42193,
    "Sofia":       61661,
    "Zagreb":      56301,
    "Nicosia":     75559,
    "Prague":      48136,
    "Copenhagen":  42786,
    "Tallinn":     38257,
    "Helsinki":    41617,
    "Paris":       44508,
    "Berlin":      45949,
    "Athens":      71611,
    "Budapest":    55973,
    "Dublin":      40665,
    "Rome":        65848,
    "Riga":        40906,
    "Vilnius":     43306,
    "Luxembourg":  45318,
    "Valletta":    75792,
    "Amsterdam":   43290,
    "Warsaw":      46826,
    "Lisbon":      70089,
    "Bucharest":   59004,
    "Bratislava":  54233,
    "Ljubljana":   52155,
    "Madrid":      69845,
    "Stockholm":   40420,
    "London":      42100,  # estimated — verify against your Table C1
}

# 50 MW plant -> specific yield kWh/kWp/year
PLANT_CAPACITY_MW = 50.0


# -------------------------------------------------------------
# 3. PVGIS API CALL
# -------------------------------------------------------------
PVGIS_URL = "https://re.jrc.ec.europa.eu/api/v5_3/PVcalc"


def query_pvgis(lat, lon, tilt, system_loss=14.0, peakpower=1.0):
    """
    Query PVGIS PVcalc endpoint for a fixed-tilt, south-facing,
    crystalline-silicon system. Returns annual yield in kWh/kWp/year.
    """
    params = {
        "lat":            lat,
        "lon":            lon,
        "peakpower":      peakpower,        # kWp
        "loss":           system_loss,      # % system losses
        "pvtechchoice":   "crystSi",
        "mountingplace":  "free",           # ground-mounted free standing
        "angle":          tilt,             # degrees
        "aspect":         0,                # south-facing (PVGIS convention)
        "raddatabase":    "PVGIS-SARAH3",
        "outputformat":   "json",
    }
    r = requests.get(PVGIS_URL, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()
    yield_kwh = data["outputs"]["totals"]["fixed"]["E_y"]   # kWh/year for 1 kWp
    return yield_kwh, data


# -------------------------------------------------------------
# 4. MAIN LOOP
# -------------------------------------------------------------
def main(output_csv="pvgis_calibration_28cities.csv", delay_s=0.5):
    # Safety check: refuse to overwrite an existing CSV without explicit confirmation
    output_path = Path(output_csv)
    if output_path.exists():
        response = input(
            f"\n⚠️  {output_csv} already exists.\n"
            f"   This script must NOT be re-run after applying f_cal to the pipeline.\n"
            f"   Are you sure you want to regenerate it? [yes/NO]: "
        )
        if response.strip().lower() != "yes":
            print("Aborted. Existing CSV preserved.")
            return None

    rows = []
    for country, city, lat, lon in CITIES:
        tilt = round(0.76 * lat + 3.1, 1)   # paper formula, rounded to 0.1
        try:
            pvgis_yield, _ = query_pvgis(lat, lon, tilt)
        except Exception as e:
            print(f"[WARN] {city}: {e}")
            pvgis_yield = None

        model_mwh = MODEL_YIELDS_MWh.get(city)
        model_yield = (model_mwh / PLANT_CAPACITY_MW) if model_mwh else None

        k_cal = (pvgis_yield / model_yield) if (pvgis_yield and model_yield) else None

        rows.append({
            "country":           country,
            "city":              city,
            "latitude":          lat,
            "longitude":         lon,
            "tilt_deg":          tilt,
            "model_yield_kWh_per_kWp": round(model_yield, 1) if model_yield else None,
            "pvgis_yield_kWh_per_kWp": round(pvgis_yield, 1) if pvgis_yield else None,
            "k_cal":             round(k_cal, 4) if k_cal else None,
        })
        print(f"{city:14s}  tilt={tilt:5.1f}  model={model_yield:7.1f}  "
              f"pvgis={pvgis_yield:7.1f}  k_cal={k_cal:.4f}")
        time.sleep(delay_s)   # polite delay between API calls

    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    print(f"\n>>> Saved: {output_csv}\n")

    # Summary stats
    print("\n--- Summary of k_cal across 28 cities ---")
    print(f"  Median: {df['k_cal'].median():.4f}")
    print(f"  Mean:   {df['k_cal'].mean():.4f}")
    print(f"  Min:    {df['k_cal'].min():.4f}  ({df.loc[df['k_cal'].idxmin(), 'city']})")
    print(f"  Max:    {df['k_cal'].max():.4f}  ({df.loc[df['k_cal'].idxmax(), 'city']})")
    print(f"  Std:    {df['k_cal'].std():.4f}")
    return df


if __name__ == "__main__":
    df = main()

Vienna          tilt= 39.7  model= 1052.8  pvgis= 1175.9  k_cal=1.1169
Brussels        tilt= 41.7  model=  843.9  pvgis= 1047.9  k_cal=1.2418
Sofia           tilt= 35.6  model= 1233.2  pvgis= 1311.2  k_cal=1.0633
Zagreb          tilt= 37.9  model= 1126.0  pvgis= 1234.6  k_cal=1.0964
Nicosia         tilt= 29.8  model= 1511.2  pvgis= 1631.4  k_cal=1.0796
Prague          tilt= 41.2  model=  962.7  pvgis= 1093.7  k_cal=1.1360
Copenhagen      tilt= 45.4  model=  855.7  pvgis= 1051.4  k_cal=1.2286
Tallinn         tilt= 48.3  model=  765.1  pvgis=  950.0  k_cal=1.2417
Helsinki        tilt= 48.8  model=  832.3  pvgis=  970.6  k_cal=1.1662
Paris           tilt= 40.2  model=  890.2  pvgis= 1130.6  k_cal=1.2701
Berlin          tilt= 43.0  model=  919.0  pvgis= 1051.0  k_cal=1.1436
Athens          tilt= 32.0  model= 1432.2  pvgis= 1574.6  k_cal=1.0994
Budapest        tilt= 39.2  model= 1119.5  pvgis= 1250.1  k_cal=1.1167
Dublin          tilt= 43.6  model=  813.3  pvgis=  987.1  k_cal=1.2137
Rome  